# 01 — Fichiers et `pathlib`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- lire et écrire un fichier texte avec `open` et `with` ;
- utiliser les **modes** (`'r'`, `'w'`, `'a'`) ;
- comprendre pourquoi **toujours** spécifier l'encodage (`encoding='utf-8'`) ;
- utiliser `Path.read_text` et `Path.write_text` pour les cas simples ;
- itérer ligne par ligne sans charger tout le fichier.

## Prérequis

- toutes les notions précédentes ;
- `pathlib`, exceptions.

Pas encore vus :

- CSV (notebook suivant), JSON.

## Plan

1. `open` et le contexte `with`
2. Modes de fichier
3. Encodage explicite
4. Lecture : `read`, `readlines`, itération
5. Écriture : `write`, `writelines`
6. `Path.read_text` / `Path.write_text`
7. Gestion d'erreurs classiques
8. Synthèse
9. Exercices

---


## 1. `open` et le contexte `with`

`open(chemin, mode)` ouvre un fichier et renvoie un objet fichier. On l'utilise **toujours** dans un `with` pour garantir la fermeture automatique.

In [ ]:
from pathlib import Path
chemin = Path('/tmp/exemple.txt')

with open(chemin, 'w', encoding='utf-8') as f:
    f.write('Bonjour, monde !\n')
    f.write('Deuxième ligne.\n')

In [ ]:
with open(chemin, 'r', encoding='utf-8') as f:
    contenu = f.read()
print(contenu)

---


## 2. Modes de fichier

| Mode | Effet | Si existe | Si absent |
|---|---|---|---|
| `'r'` | lecture (par défaut) | lit | `FileNotFoundError` |
| `'w'` | écriture | **écrase** | crée |
| `'a'` | ajout | ajoute à la fin | crée |
| `'x'` | création exclusive | `FileExistsError` | crée |
| `'rb'` / `'wb'` | binaire | idem | idem |

**Pour du texte, ne pas mettre de `b`.**

In [ ]:
from pathlib import Path
chemin = Path('/tmp/exemple.txt')

with open(chemin, 'a', encoding='utf-8') as f:
    f.write('Ligne ajoutée.\n')

print(chemin.read_text(encoding='utf-8'))

---


## 3. Encodage explicite

**Toujours** spécifier `encoding='utf-8'`. Sur Windows, l'encodage par défaut dépend de la locale — ce qui crée des bugs invisibles sur les autres machines.

---


## 4. Lecture

### `read()` — tout le fichier en une chaîne

In [ ]:
from pathlib import Path
chemin = Path('/tmp/exemple.txt')

with open(chemin, 'r', encoding='utf-8') as f:
    contenu = f.read()

print(len(contenu), 'caractères')

### `readlines()` — liste de lignes (avec `\n` à la fin)

In [ ]:
with open(chemin, 'r', encoding='utf-8') as f:
    lignes = f.readlines()

print(lignes)

### Itération directe — idiomatique et économe

In [ ]:
with open(chemin, 'r', encoding='utf-8') as f:
    for ligne in f:
        print(repr(ligne))

L'itération directe **ne charge pas tout le fichier** en mémoire. Préférer cette forme pour les gros fichiers.

---


## 5. Écriture

In [ ]:
lignes = ['un\n', 'deux\n', 'trois\n']
with open('/tmp/nombres.txt', 'w', encoding='utf-8') as f:
    f.writelines(lignes)

print(open('/tmp/nombres.txt', encoding='utf-8').read())

⚠️ `writelines` n'ajoute **pas** de `\n` automatiquement. C'est à vous de le faire.

---


## 6. `Path.read_text` / `Path.write_text`

Pour les cas simples (tout lire, tout écrire), `pathlib` offre des raccourcis plus lisibles.

In [ ]:
from pathlib import Path
p = Path('/tmp/rapide.txt')
p.write_text('Hello world\n', encoding='utf-8')

In [ ]:
p.read_text(encoding='utf-8')

⚠️ `write_text` **écrase** le fichier. Pour ajouter, il faut passer par `open('a')`.

---


## 7. Gestion d'erreurs

In [ ]:
try:
    with open('/inexistant.txt', 'r', encoding='utf-8') as f:
        f.read()
except FileNotFoundError as err:
    print('fichier introuvable :', err)

In [ ]:
try:
    with open('/root/protege.txt', 'w', encoding='utf-8') as f:
        f.write('test')
except PermissionError as err:
    print('droits insuffisants :', err)

---


## 8. Synthèse

| Besoin | Forme |
|---|---|
| Ouvrir en lecture | `open(p, 'r', encoding='utf-8')` |
| Lire tout | `f.read()` ou `Path.read_text()` |
| Itérer ligne par ligne | `for ligne in f:` |
| Écrire (écraser) | `open(p, 'w', ...)` ou `Path.write_text()` |
| Ajouter à la fin | `open(p, 'a', ...)` |
| Créer sans écraser | `open(p, 'x', ...)` |

### Règles

1. Toujours `with`.
2. Toujours `encoding='utf-8'`.
3. Itérer directement pour les gros fichiers.
4. `Path.read_text` / `write_text` pour le code concis.
5. Attraper `FileNotFoundError`, `PermissionError`, `IsADirectoryError` selon le contexte.

---


## 9. Exercices

### Exercice 1 — Écrire et relire *(facile)*

Écrire `sauver_et_relire(contenu: str, chemin: str) -> str` qui écrit `contenu` dans `chemin` puis le relit et le renvoie.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Fichiers_et_pathlib", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from pathlib import Path

def sauver_et_relire(contenu: str, chemin: str) -> str:
    """Écrit puis relit le contenu."""
    p = Path(chemin)
    p.write_text(contenu, encoding='utf-8')
    return p.read_text(encoding='utf-8')

print(sauver_et_relire('hello', '/tmp/demo_sauver.txt'))
```

</details>

### Exercice 2 — Compter les lignes *(facile)*

Écrire `compter_lignes(chemin: str) -> int` qui renvoie le nombre de lignes du fichier.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Fichiers_et_pathlib", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
def compter_lignes(chemin: str) -> int:
    """Nombre de lignes d'un fichier texte."""
    total = 0
    with open(chemin, 'r', encoding='utf-8') as f:
        for _ in f:
            total = total + 1
    return total

# préparer un fichier pour tester
from pathlib import Path
Path('/tmp/demo_lignes.txt').write_text('un\ndeux\ntrois\n', encoding='utf-8')
print(compter_lignes('/tmp/demo_lignes.txt'))
```

</details>

### Exercice 3 — Ajouter une ligne *(facile)*

Écrire `ajouter_ligne(chemin: str, ligne: str) -> None` qui ajoute `ligne` à la fin du fichier (avec `\n`). Utiliser le mode `'a'`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Fichiers_et_pathlib", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
def ajouter_ligne(chemin: str, ligne: str) -> None:
    """Ajoute une ligne à la fin du fichier."""
    with open(chemin, 'a', encoding='utf-8') as f:
        f.write(ligne + '\n')

from pathlib import Path
Path('/tmp/demo_append.txt').write_text('initial\n', encoding='utf-8')
ajouter_ligne('/tmp/demo_append.txt', 'ajouté')
print(Path('/tmp/demo_append.txt').read_text(encoding='utf-8'))
```

</details>

### Exercice 4 — Lecture tolérante *(moyen)*

Écrire `lire_ou_defaut(chemin: str, defaut: str) -> str` qui renvoie le contenu du fichier, ou `defaut` si le fichier n'existe pas.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Fichiers_et_pathlib", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
def lire_ou_defaut(chemin: str, defaut: str) -> str:
    """Lit le fichier ou renvoie defaut si absent."""
    try:
        with open(chemin, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return defaut

print(lire_ou_defaut('/absent.txt', 'VIDE'))
```

</details>

### Exercice 5 — Inverser les lignes *(moyen)*

Écrire `inverser_lignes(chemin: str) -> None` qui lit un fichier et le réécrit avec les lignes dans l'ordre inverse.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Fichiers_et_pathlib", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
def inverser_lignes(chemin: str) -> None:
    """Réécrit le fichier en inversant l'ordre des lignes."""
    with open(chemin, 'r', encoding='utf-8') as f:
        lignes = f.readlines()
    with open(chemin, 'w', encoding='utf-8') as f:
        f.writelines(reversed(lignes))

from pathlib import Path
Path('/tmp/demo_inv.txt').write_text('a\nb\nc\n', encoding='utf-8')
inverser_lignes('/tmp/demo_inv.txt')
print(Path('/tmp/demo_inv.txt').read_text(encoding='utf-8'))
```

</details>

### Exercice 6 — Compter les mots *(moyen)*

Écrire `compter_mots(chemin: str) -> dict[str, int]` qui renvoie un dict `{mot: nombre}` pour tout le fichier (découper avec `.split()`).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Fichiers_et_pathlib", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
def compter_mots(chemin: str) -> dict[str, int]:
    """Compte les mots d'un fichier texte."""
    freq: dict[str, int] = {}
    with open(chemin, 'r', encoding='utf-8') as f:
        for ligne in f:
            for mot in ligne.split():
                freq[mot] = freq.get(mot, 0) + 1
    return freq

from pathlib import Path
Path('/tmp/demo_mots.txt').write_text('un deux un\ntrois un deux\n', encoding='utf-8')
print(compter_mots('/tmp/demo_mots.txt'))
```

</details>

---


## Ressources externes

- [Lecture/écriture — tutoriel](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)
- [`open()` — doc](https://docs.python.org/3/library/functions.html#open)
- [`pathlib`](https://docs.python.org/3/library/pathlib.html)
- [PEP 597 — `encoding` explicite](https://peps.python.org/pep-0597/)

---

## Mini-exemples supplémentaires

### Vérifier l'existence

In [ ]:
from pathlib import Path
print(Path('/tmp').exists())
print(Path('/inexistant').exists())

### Créer un dossier

In [ ]:
from pathlib import Path
p = Path('/tmp/demo_dir')
p.mkdir(exist_ok=True)
print(p.is_dir())

### `Path.glob` pour lister selon un motif

In [ ]:
from pathlib import Path
# Liste les .py du dossier /tmp/demo_dir (s'il existe)
for p in Path('/tmp').glob('*.txt'):
    print(p.name)

### Taille d'un fichier

In [ ]:
from pathlib import Path
p = Path('/tmp/demo_taille.txt')
p.write_text('hello world\n', encoding='utf-8')
print(p.stat().st_size, 'octets')

### Chemin absolu / résolu

In [ ]:
from pathlib import Path
p = Path('./chemin/../relatif')
print(p)
print(p.resolve())

### Renommer, déplacer

In [ ]:
from pathlib import Path
src = Path('/tmp/a_renommer.txt')
src.write_text('hi', encoding='utf-8')
dest = Path('/tmp/renomme.txt')
src.replace(dest)
print(dest.read_text(encoding='utf-8'))

### Supprimer

In [ ]:
from pathlib import Path
p = Path('/tmp/a_supprimer.txt')
p.write_text('bye', encoding='utf-8')
p.unlink()
print(p.exists())

### Binaire

In [ ]:
from pathlib import Path
p = Path('/tmp/demo.bin')
p.write_bytes(b'\x00\x01\x02\x03')
print(p.read_bytes())